# Stage A, part 3 - scoring the three students on the validation splits

The students F, N and C of notebook 02 are scored with the run step of part two, unchanged, as part three's notebook 03 ran it. Frozen features, a linear probe and the small decoder with the seeds 0, 1 and 2, the same metrics, the same four SAR datasets and the same aerial set. In stage A only the validation splits are scored. The test splits wait for the final students of stage B.

The decoder trains with part two's step, as every saved row. A compiled step (torch.compile) was tried and is off. It needs 39 and 38 percent less time per step at 512 and 992 px, but it changed the result of a seed. The water IoU of the forest control on val moved 0.020 down in the first run of this notebook and 0.013 up in notebook 03a, both times the one number of 22 outside the control's rule. FAST_DECODER in the setup cell keeps the switch.

## What is scored

- the control first. The forest run of part two (satellite DINOv3 at layer 21, the three channel input) is repeated and compared with part two's saved rows the way part three's control did it, every shared number printed side by side with its gap. The majority and probe rows are deterministic, a score column may differ by at most 0.02 and a count column (pixels, layer) by at most 1e-6. The decoder rows are not, part two trained its decoder under bfloat16 autocast on CUDA, so the decoder scores are compared by the mean over the seeds with the tie rule, a gap of at most 0.01 or at most twice the larger seed spread passes. Anything else stops the notebook
- the four SAR datasets of part two (glacial lakes, snow, forest, alpine glaciers) with the students F, N and C in Sentinel-1 mode (two channels, standardised with the training split of each dataset)
- the aerial validation set of FLAIR-HUB (2,000 patches of the test departments, 19 classes) with the three students in aerial mode. The probe set of 4,000 patches is the training split

## Saved rows, never run again

- part two - the satellite DINOv3 ViT-L at layer 21 and C-RADIOv4-H at layer 32, from part two's results folder on Drive
- part three - TerraFM-B reading Sentinel-1, the pilot students M and S, and DINOv3 SAT reading the aerial RGB, from part three's results folder, checked against part three's own stage A table

Every saved row is read from the summary file of its own run, so another layer or mode in the same folder never mixes in. Nothing is written into the folders of part two or part three.

## Table and verdicts

One table of every dataset, head and model on the validation split. Lakes, alpine glaciers and snow are read by the IoU of the positive class, forest and the aerial set by mIoU, as part three's table. Then the verdicts, written by the code, not by hand

- the rule per row. Decoder, a win when the mean is higher by more than 0.01 AND by more than twice the larger seed SD, a loss the same way round, else a tie. Probe, a win or a loss when the gap is more than 0.02
- counted over the 4 SAR validation decoder rows. The probe rows are tallied next to them, the aerial row is reported and not counted
- the stage A words. 4 wins and no loss = beats, no loss and fewer than 4 wins = matches, 1 or more losses and more wins than losses = mixed, otherwise does not reach

for the questions

1. F against the frozen satellite DINOv3, the headline
2. F against C, whether the radar teacher helps
3. N against F, SigLIP2 NaFlex against SigLIP2 FixRes
4. F and N against their radar teacher, TerraFM-B reading Sentinel-1
5. F, N and C against student M of the part three pilot, informational

## Data and the .env file

The .env file says where everything is, the notebook never names another project's folders itself.

- DATASET_HF_OWNER, the Hugging Face owner of the private copies of the four SAR datasets. Without it, or when a fetch fails, the copies on Drive under MyDrive/datasets are used. Only the splits read here are taken, train and val, and every split of the forest dataset for the control
- DATA_HF_REPO and DATA_DRIVE_DIR, the pilot data of part three. The aerial probe and validation sets come from the private repo first and from the archives on Drive when the fetch fails, the fetch prints which source was used. The sets are read straight out of the archives, nothing is unpacked. Part three's results folder is in DATA_DRIVE_DIR too
- REFERENCE_DRIVE_DIR, the folder under MyDrive that holds part two's results
- HF_TOKEN, the read token. It fetches the private copies and the gated DINOv3 weights of the control. It is never printed

## Data in the background

Every download starts right after the setup cell, in one background fetcher with two lanes of one thread each, and no cell waits for data it does not need yet.

- the data lane takes one item at a time, so the item needed first gets the whole bandwidth (a dataset fetch already runs 8 files at once from the Hub or 4 from Drive). Forest first, the control needs it. Then the alpine glaciers, small (about 1.2 GB of train and val) but with the longest runs, so once they are on disk the GPU has work for longer than the rest takes to arrive. Then lakes (about 5.5 GB) before snow (about 7 GB), and the aerial sets last, their runs come after every SAR run
- the read-ahead lane reads the DINOv3 weights of the control (1.2 GB) and the three student files once from Drive at the same time (with FAST_DECODER it first unpacks the compile cache of an earlier session), so the control and the student processes find them in the page cache
- the control waits for forest and the DINOv3 weights only. The student runs of a dataset start as soon as it is on the runtime disk, every dataset there goes into one call and the ones still arriving into the next, so the GPU waits only when none of the datasets left is there. The aerial runs wait for the aerial sets
- a fetch that fails falls back to the copy on Drive without stopping the others, a dataset on neither is named and its runs are MISSING. A dataset whose student runs all have their summary on Drive is not fetched, forest always (the control reads it)
- what a fetch prints shows in the cell that waits for it, with the seconds from the start to when it was ready, its MB per second and the seconds that cell waited. The aerial runs cell ends with every item and the seconds the GPU work waited for data against the data one after the other

## Runtime

The A100 with High-RAM, as part three's notebook 03. The decoder at 992 px and the features of the large tiles need the GPU and the RAM, and with the three students side by side the cells estimate up to about 60 GB of GPU memory and 100 GB of RAM at once (estimates that err high, printed per dataset). On an A100 with 40 GB fewer students run at once. The notebook is opened from VS Code with the Colab extension.

## Run time

About 1 hour 40 minutes, an estimate. Part three measured 16 scoring runs in 121 minutes on an A100 with 80 GB, one run after another (the alpine decoder at 992 px about 21 minutes per encoder, the others 2 to 8). Here there are 16 runs too, 3 students x 5 datasets plus the control, with students of the same size. The three students run side by side, one process each on the same GPU. That pays where one run leaves the GPU idle, the small tiles of lakes, forest and the aerial set. The decoders at 512 and 992 px keep the GPU busy on their own (their step time grows with the pixels in part three), so snow and the alpine glaciers gain little. The data no longer comes before the runs. About 14 GB of SAR data and 2 GB of aerial archives arrive while the GPU works (part three's data cells took 2.4 minutes from the Hub before its first run, the copies on Drive take longer), and the GPU waits only at the start, for forest and the DINOv3 weights of the control, which the old control read itself. Part two's step took 115 ms at 992 px and 32 ms at 512 px on the A100 in notebook 03a, as in part three. The run cells print the wall time against the runs added up and against part three's minutes one run at a time. Every cell prints its time, every run its minutes, every fetch and read its MB per second.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/results/. The files of part two's run step per dataset and student (summary, per region, the trained heads, the info), the tables (stage_a_table.csv every row, stage_a_val_table.csv by mIoU, stage_a_val_table_pos.csv by the reading score, stage_a_rows.csv every compared row with its outcome) and verdicts.json.

## How to run

Run the cells from top to bottom. A dataset already on the runtime disk and a run whose summary is on Drive are skipped, so after a crash run the cells again from the top, the fetcher of the first run stops before the new one starts. The control runs once, on a new runtime its saved summary is compared again. A run that fails is printed as FAILED with its student and dataset, the other runs go on, and the cell ends with the runs without a summary on Drive. Everything the run step prints is in one log per student, results/logs/score__student-f.log and so on. The last cell flushes Drive and then releases the Colab machine, so the GPU stops by itself when the run is done (set END_RUNTIME = False in that cell to keep the runtime), and to use Drive afterwards run the mount cell again.

In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Tokens, data locations, model cache, GPU check, folders, the three students and the saved rows on Drive.
import json
import os
import shutil
import tarfile
import time
from pathlib import Path

import pandas as pd
import torch

from sartransfer import env
from sartransfer.distil import aerial, score, stage0, train
from sartransfer.domains import aerial as aerial_domain, hub, prepared
from sartransfer.domains import run as run_step
from sartransfer.domains.prepared import load_manifest
from sartransfer.models import LAST

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the A100 runtime with High-RAM and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private data copies")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as part two's run notebook
DATA_DIR = env.require("DATA_DRIVE_DIR", "the folder under MyDrive with the pilot data and part three's results")
REFERENCE_DIR = env.require("REFERENCE_DRIVE_DIR", "the folder under MyDrive with part two's results")
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copies are used

# Settings of this notebook. The names are those of the code.
SAR_DATASETS = score.SAR_DATASETS              # glacial_lakes, snow, forty_v1, glavitu_alp, the 4 rows of a word
AERIAL_DATASET = score.AERIAL_DATASET          # flair_hub, reported next to them
SPLITS = ("val",)                              # stage A, the validation splits only
STUDENTS = {"student-f": "student_F", "student-n": "student_N", "student-c": "student_C"}   # encoder name and
                                               # the folder notebook 02 wrote it to
SAR_ENCODERS = AERIAL_ENCODERS = tuple(STUDENTS)
S1_MODE, AERIAL_MODE = "co_cross", "aerial_rgbn"   # how the students and part three's encoders read the data
PART2_ENCODERS, PART2_MODE = ("dinov3-l-sat", "cradio-v4-h"), "vv_vh_diff"   # part two's saved rows, each at
                                               # its pre-registered layer (21 and 32)
PART3_SAR = ("terrafm-b-s1", "student-m", "student-s")          # part three's saved rows, final output
PART3_AERIAL = ("student-m", "student-s", "dinov3-aerial")
POSITIVE_CLASS_DATASETS = score.POSITIVE_CLASS_DATASETS   # read by the IoU of the positive class, as part
                                               # three's table; forest and the aerial set by mIoU
CONTROL_DATASET, CONTROL_ENCODER = "forty_v1", "dinov3-l-sat"
SCORE_TOLERANCE = 0.02                         # control, a score column (IoU, mIoU, accuracy, F1, ...)
COUNT_TOLERANCE = 1e-6                         # control, a count column (n_pixels, layer)
DECODER_MEAN_TOLERANCE = 0.01                  # control, decoder scores by seed mean, the tie rule
SAVED_TOLERANCE = 1e-9                         # part three's summaries against part three's own table
FAST_DECODER = False                           # part two's decoder step. The compiled one needs 39 and 38 percent
                                               # less time at 512 and 992 px but changed the result of a seed, water IoU
                                               # on val moved 0.020 down in the first control and 0.013 up in
                                               # notebook 03a, by the rule written before it

CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PREP = CONTENT / "sar-transfer-distillation" / "prepared"      # runtime disk, a few large files per dataset
HUB_DIR = CONTENT / "sar-transfer-distillation" / "hub"        # the aerial sets fetched from the pilot repo
RES = MYDRIVE / "sar-transfer-distillation" / "results"        # the results of this notebook
COMPILE_CACHE = CONTENT / "sar-transfer-distillation" / "compile_cache"   # the compiled decoders of this session
COMPILE_ARCHIVE = MYDRIVE / "sar-transfer-distillation" / "compile_cache.tar"   # the same as one file on Drive, so a
                                               # later session reads the compiled decoders instead of compiling
RUNS = MYDRIVE / "sar-transfer-distillation" / f"runs_{train.TrainConfig().steps}"   # the students of notebook 02,
                                               # one folder per schedule, named by the steps
REF = MYDRIVE / REFERENCE_DIR / "results"               # part two's saved results, read only
PART3 = MYDRIVE / DATA_DIR / "results"                  # part three's saved results, read only
DRIVE_AERIAL = MYDRIVE / DATA_DIR / "pilot" / "aerial"  # the aerial archives of the pilot data
DRIVE_DATASETS = MYDRIVE / "datasets"                   # the Drive copies of the SAR datasets, one folder each
CHECKPOINTS = {name: RUNS / folder / train.FINAL for name, folder in STUDENTS.items()}
SAVED_FOLDERS = {"part two": REF, "part three": PART3}
SAVED_RUNS = {
    "part two": [score.run_name(d, e, run_step.PRE_REGISTERED_LAYERS[e], PART2_MODE)
                 for e in PART2_ENCODERS for d in SAR_DATASETS],
    "part three": ([score.run_name(d, e, LAST, S1_MODE) for e in PART3_SAR for d in SAR_DATASETS]
                   + [score.run_name(AERIAL_DATASET, e, LAST, AERIAL_MODE) for e in PART3_AERIAL])}
SAR_RUNS = [score.run_name(d, e, LAST, S1_MODE) for e in SAR_ENCODERS for d in SAR_DATASETS]
AERIAL_RUNS = [score.run_name(AERIAL_DATASET, e, LAST, AERIAL_MODE) for e in AERIAL_ENCODERS]

for name, path in CHECKPOINTS.items():
    assert path.is_file(), f"{path} is not on Drive, run notebook 02 first"
    print(f"{name:<10} {path}")
for source, folder in SAVED_FOLDERS.items():
    assert folder.is_dir(), f"{folder} is not on Drive, it holds the saved rows of {source}"
    absent = [run for run in SAVED_RUNS[source] if not (folder / f"{run}__summary.csv").is_file()]
    assert not absent, f"the saved rows of {source} are not all in {folder}, missing {absent}"
    print(f"saved rows of {source:<10} {len(SAVED_RUNS[source])} runs in {folder}")
assert (PART3 / "stage_a_table.csv").is_file(), f"{PART3 / 'stage_a_table.csv'} is not on Drive, part three's own table"
assert RES not in SAVED_FOLDERS.values(), "the results of this notebook would go into a folder of the saved rows"
for folder in (PREP, RES, HUB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

TIMES = {}                       # seconds of every step of this run, by name


def num(value, digits=2):
    """A number as text, "not known" for None and nan."""
    if value is None or value != value:
        return "not known"
    return f"{value:,.{digits}f}"


def show(table):
    """Print a whole table."""
    print(table.to_string(index=False))


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def took(since, name):
    """The time of a cell as text, recorded under `name` for the summary."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def missing_runs(runs):
    """The runs of this notebook without a summary on Drive."""
    return [run for run in runs if not (RES / f"{run}__summary.csv").is_file()]


runtime = stage0.runtime_report([PREP, RES])
print(f"GPU     {runtime['gpu']}, {num(runtime['gpu_memory_gb'], 1)} GB, "
      f"free {num(runtime['gpu_free_gb'], 1)} GB")
print(f"RAM     {num(runtime['ram_gb'], 1)} GB, free {num(runtime['ram_free_gb'], 1)} GB, "
      f"{runtime['cpus']} CPUs")
for folder, disk in runtime["disk"].items():
    print(f"disk    {folder}, free {num(disk['free_gb'], 1)} of {num(disk['total_gb'], 1)} GB")
print(f"results go to   {RES}")
print(f"prepared data   {PREP}")
print(f"SAR datasets    {'Hugging Face, then Drive' if DATASET_OWNER else 'Drive'} ({DRIVE_DATASETS})")
stage0.save_report(RES, runtime=runtime)
print(took(start, "setup"))

In [ ]:
# Prefetch. Every download starts here, in one background fetcher (score.Prefetch), and each cell below waits
# only for its own items. Two lanes, one thread each. The data lane takes one item at a time - a dataset fetch
# already runs hub.DATASET_FETCH_WORKERS files at once from the Hub or prepared.COPY_WORKERS from Drive, so the
# item needed first gets the whole bandwidth instead of sharing it with the next ones. The read-ahead lane reads
# the weight files of the control and of the students once over the Drive mount at the same time, into the page
# cache (score.read_ahead, one thread per file, the reason is written there). The SAR datasets come from their
# private Hugging Face copies when DATASET_HF_OWNER is set, else (and when a fetch fails) from their copies on
# Drive under MyDrive/datasets, only the splits read here, train and val, and every split of the control dataset,
# whose saved rows hold val and test. The aerial probe and validation sets come from the private pilot repo
# (DATA_HF_REPO), else from their archives on Drive, read straight out of the archives (nothing unpacked) and
# written once in the prepared format of part two. A dataset on the runtime disk is not fetched again, one with
# files missing (a fetch that was cut) is taken again, one whose student runs all have their summary on Drive is
# not fetched at all (forest always, the control reads it). What a fetch prints, the source it used and its MB
# per second, is printed by the cell that waits for it.
start = time.perf_counter()
if globals().get("prefetch") is not None:     # a second run in this session, the fetcher of the first stops first
    prefetch.close()
from huggingface_hub.utils import disable_progress_bars

from sartransfer.models.encoders import ENCODERS

disable_progress_bars()                       # a bar of a download in the background would land in whatever cell runs
NEEDED = {name: None if name == CONTROL_DATASET else ("train", *SPLITS) for name in SAR_DATASETS}   # None = all
# The order of the data lane. Forest first, the control needs it (its DINOv3 weights come at the same time on the
# read-ahead lane). Then the alpine glaciers - small (236 train and val chips of 992 px, about 1.2 GB) but with the
# longest runs (about 22 min each one at a time in part three), so once they are on disk the GPU has work for longer
# than the rest takes to arrive. Then lakes (about 5.5 GB) before snow (about 7 GB, the biggest), the smaller one
# first. Snow last of the SAR sets cannot make a run wait, the forest and alpine runs alone take longer than its
# fetch. The aerial sets last, their runs come after every SAR run. The student weights follow the DINOv3 weights,
# the student runs start after the control
DATA_ORDER = ("forty_v1", "glavitu_alp", "glacial_lakes", "snow")
assert sorted(DATA_ORDER) == sorted(SAR_DATASETS), "the data lane holds every SAR dataset once"
CONTROL_RUN = score.run_name(CONTROL_DATASET, CONTROL_ENCODER, run_step.PRE_REGISTERED_LAYERS[CONTROL_ENCODER],
                             PART2_MODE)


def on_disk(name):
    """True when the manifest and every file of the splits read here are on the runtime disk."""
    if not (PREP / name / "manifest.json").is_file():
        return False
    splits = NEEDED[name] or load_manifest(PREP, name)["splits"]
    return all((PREP / name / file).is_file() for file in prepared.split_files(splits))


def fetch_sar(name):
    """One SAR dataset to the runtime disk, from its private Hugging Face copy when DATASET_HF_OWNER is set, else
    (and when that fails) from its copy on Drive. Raises when it is on neither."""
    if on_disk(name):
        return {"source": "on the runtime disk already"}
    if (PREP / name / "manifest.json").is_file():
        print(f"{name}  incomplete on disk, taken again")
    clock, source = time.perf_counter(), None
    if DATASET_OWNER:
        try:                                       # first the private Hugging Face copy
            hub.fetch(name, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=NEEDED[name])
            source = "Hugging Face"
        except Exception as err:
            print(f"{name}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
    if source is None:                             # then the copy on Drive
        copy = DRIVE_DATASETS / hub.repo_name(name)
        if not (copy / "manifest.json").is_file():
            raise FileNotFoundError(f"neither on Hugging Face nor on Drive under {copy}")
        prepared.copy_prepared(copy, PREP, name, splits=NEEDED[name])
        source = "Drive"
    man = load_manifest(PREP, name)
    print(f"{name}  from {source} in {(time.perf_counter() - clock) / 60:.1f} min")
    print(f"{name:<14} splits {man['splits']}, {man['chip_size']} px, units {man['units']}, free disk "
          f"{shutil.disk_usage(PREP).free / stage0.GB:.0f} GB")
    files = prepared.split_files(NEEDED[name] or man["splits"])
    return {"source": source, "bytes": sum((PREP / name / file).stat().st_size for file in files)}


def fetch_aerial():
    """The aerial probe and validation sets of FLAIR-HUB in the prepared format of part two on the runtime disk,
    from the private pilot repo, else from their archives on Drive. The probe set is the training split."""
    if (PREP / AERIAL_DATASET / "manifest.json").exists():
        return {"source": "on the runtime disk already"}
    clock = time.perf_counter()
    try:                                           # first the private Hugging Face repo, hub.FETCH_WORKERS files at once
        fetched = hub.fetch_pilot(HUB_DIR, token=HF_TOKEN,
                                  want=("aerial/manifest.json", "aerial/probe.tar", "aerial/val.tar"))
        folder, source = HUB_DIR / "aerial", "the private Hugging Face repo of DATA_HF_REPO"
        print(f"aerial sets from the private Hugging Face repo of DATA_HF_REPO, {len(fetched)} files, "
              f"{(time.perf_counter() - clock) / 60:.1f} min")
    except Exception as err:                       # then the archives on Drive
        folder, source = DRIVE_AERIAL, "the archives on Drive"
        print(f"aerial sets from the archives on Drive, not from Hugging Face ({type(err).__name__}, "
              f"{hide(err)[:200]})")
    manifest_file = folder / "manifest.json"
    assert manifest_file.is_file(), f"{manifest_file} is missing, the pilot data of part three holds it"
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    sets, archive_bytes = {}, 0
    for name in ("probe", "val"):                  # the npz members straight out of the tar, nothing unpacked
        archive = folder / f"{name}.tar"
        assert archive.is_file(), f"{archive} is missing, the pilot data of part three holds it"
        clock = time.perf_counter()
        try:
            sets[name] = aerial.load_set_from_tar(archive, name, manifest["sets"][name]["files"])
        except FileNotFoundError as err:          # a file of the manifest is not in the tar
            raise AssertionError(hide(err)) from None
        except (tarfile.TarError, EOFError, OSError) as err:
            raise RuntimeError(f"{archive} cannot be read, {type(err).__name__}, {hide(err)}") from None
        seconds = time.perf_counter() - clock
        archive_bytes += archive.stat().st_size
        mb = archive.stat().st_size / stage0.MB
        print(f"{name:<6} read in place from {archive.name}  {sets[name]['images'].shape}, "
              f"{len(set(sets[name]['departments']))} departments, {mb:,.0f} MB in {seconds:.1f} s, "
              f"{mb / max(seconds, 1e-9):,.0f} MB per second")
    licence = {"name": "FLAIR-HUB", "provider": "IGN", "repo": aerial.HF_REPO,
               "revision": aerial.HF_REVISION, "licence": "Etalab Open Licence 2.0",
               "attribution": aerial.ATTRIBUTION}
    aerial_domain.aerial_to_prepared({"train": sets["probe"], "val": sets["val"]}, PREP,
                                     dict(enumerate(aerial.CLASSES)), dataset=AERIAL_DATASET,
                                     nodata=aerial.IGNORE, pixel_spacing_m=aerial.PIXEL_M, source=licence,
                                     notes="stage A, the probe set as train and the validation set of "
                                           "the test departments, no test set")
    man = load_manifest(PREP, AERIAL_DATASET)
    print(f"{AERIAL_DATASET:<14} splits {man['splits']}, {man['chip_size']} px, {len(man['classes'])} classes, "
          f"units {man['units']}")
    return {"source": source, "bytes": archive_bytes}


def read_control_weights():
    """The files the DINOv3 of the control reads (AutoModel.from_pretrained), from the model cache on Drive once
    into the page cache. Files not in the cache are left to the control, which reads or fetches them as before."""
    files = score.model_files(ENCODERS[CONTROL_ENCODER].model_id, cache_dir=MODEL_CACHE / "hub")
    if not files:
        raise FileNotFoundError("the DINOv3 weights are not in the model cache on Drive, the control fetches them")
    return {"source": "the model cache on Drive", **score.read_ahead(files)}


def restore_cache():
    """The compile cache archive of an earlier session from Drive into COMPILE_CACHE, before the first compile."""
    got = score.restore_compile_cache(COMPILE_ARCHIVE, COMPILE_CACHE)
    return {"source": f"Drive, {got['status']}", "bytes": int(got["mb"] * 1e6)}


def read_students():
    """The final.pt files of the three students on Drive once into the page cache, before their processes load them."""
    return {"source": "Drive", **score.read_ahead(CHECKPOINTS.values())}


SAR_TODO = [d for d in SAR_DATASETS if missing_runs([score.run_name(d, e, LAST, S1_MODE) for e in SAR_ENCODERS])]
AERIAL_TODO = bool(missing_runs(AERIAL_RUNS))
prefetch = score.Prefetch(secrets=(HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")))
for name in DATA_ORDER:
    if name == CONTROL_DATASET or name in SAR_TODO:
        prefetch.add(name, lambda name=name: fetch_sar(name), lane="data")
if AERIAL_TODO:
    prefetch.add(AERIAL_DATASET, fetch_aerial, lane="data")
CONTROL_TODO = not (RES / f"{CONTROL_RUN}__summary.csv").is_file()   # a control run that is done loads no model
if FAST_DECODER and (CONTROL_TODO or SAR_TODO or AERIAL_TODO):     # first on its lane, the student weights
    prefetch.add("compile cache", restore_cache, lane="read-ahead")   # come after it, so a run that waits
if CONTROL_TODO:                                                    # for them finds the cache unpacked
    prefetch.add("DINOv3 weights", read_control_weights, lane="read-ahead")
if SAR_TODO or AERIAL_TODO:
    prefetch.add("student weights", read_students, lane="read-ahead")
prefetch.start()
if not DATASET_OWNER:
    print("DATASET_HF_OWNER is not set, the SAR datasets come from their copies on Drive")
for lane, names in prefetch.lanes().items():
    print(f"{lane:<10} lane, one item after the other   {', '.join(names)}")
for name in (*SAR_DATASETS, AERIAL_DATASET):
    if name not in prefetch:
        print(f"{name}  every student run has its summary on Drive, not fetched")
print("started, each cell below waits only for its own items and prints them when it has them")
print(took(start, "prefetch"))

In [ ]:
# Control. The forest run of part two is repeated with the saved run step and compared with the saved
# rows of part two, every shared number side by side with its gap, as part three's control. The majority
# and probe rows are deterministic, so every number must agree within SCORE_TOLERANCE (a score) or
# COUNT_TOLERANCE (a count). The decoder rows are not, part two trained its decoder under bfloat16 autocast
# on CUDA, whose kernels are not bit-reproducible across sessions, so the decoder scores are compared the
# way every decoder comparison here works, by the mean over the seeds with the tie rule - a gap of at most
# DECODER_MEAN_TOLERANCE, or at most twice the larger seed spread, passes. Counts must agree on every row.
# Anything else stops the notebook. A control run whose summary is on Drive is not run again.
# It waits for the forest dataset and the DINOv3 weights only, the rest of the data is still arriving.
start = time.perf_counter()
forest = prefetch.wait(CONTROL_DATASET, by="control")
assert forest["ok"], f"{CONTROL_DATASET} is not on the runtime disk, the control needs it - {forest['error']}"
if "compile cache" in prefetch and CONTROL_TODO:   # the control compiles only when it runs
    prefetch.wait("compile cache", by="control")
if "DINOv3 weights" in prefetch:              # not read ahead when not in the cache, the model reads them itself
    prefetch.wait("DINOv3 weights", by="control")
layer = run_step.PRE_REGISTERED_LAYERS[CONTROL_ENCODER]
control_run = score.run_name(CONTROL_DATASET, CONTROL_ENCODER, layer, PART2_MODE)
reference_file = REF / f"{control_run}__summary.csv"
print(f"control run  {control_run}")
assert reference_file.is_file(), (f"{reference_file} is not on Drive, the control needs the saved "
                                  "rows of part two")
score.score_all(PREP, [CONTROL_ENCODER], [CONTROL_DATASET], RES, token=HF_TOKEN, device=DEVICE,
                fast_decoder=FAST_DECODER, compile_cache=COMPILE_CACHE)
repeated_file = RES / f"{control_run}__summary.csv"
assert repeated_file.is_file(), "the control run did not finish, see FAILED above"
saved, repeated = pd.read_csv(reference_file), pd.read_csv(repeated_file)
KEYS = ["head", "seed", "split"]
shared = [c for c in saved.columns if c in repeated.columns and c not in KEYS
          and pd.api.types.is_numeric_dtype(saved[c]) and pd.api.types.is_numeric_dtype(repeated[c])]
counts = [c for c in shared if pd.api.types.is_integer_dtype(saved[c])]      # n_pixels, layer
scores = [c for c in shared if c not in counts]                              # IoU, mIoU, accuracy, ...
allowed = pd.Series({c: COUNT_TOLERANCE if c in counts else SCORE_TOLERANCE for c in shared}, dtype=float)
a = saved.set_index(KEYS).sort_index()[shared]
b = repeated.set_index(KEYS).sort_index()[shared]
same_rows = a.index.equals(b.index)
if same_rows:
    both_nan = a.isna() & b.isna()
    gap = (a - b).abs().where(~both_nan, 0.0)
    decoder = (a.index.get_level_values("head") == "decoder")
    cell_allowed = pd.DataFrame([allowed.to_numpy()] * len(a), index=a.index, columns=shared)
    cell_allowed.loc[decoder, scores] = float("nan")       # judged by the seed means below, not per number
    off = gap.isna() | gap.gt(cell_allowed)
    off.loc[decoder, scores] = False
    compared = pd.concat({c: pd.DataFrame({"part two": a[c], "part four": b[c], "gap": gap[c],
                                           "allowed": cell_allowed[c], "off": off[c]}) for c in shared},
                         names=["column"])
    print(f"every shared number, part two against the repeated run, {len(compared)} numbers (decoder "
          "scores are judged by their seed means, see below)")
    print(compared.to_string(float_format=lambda v: f"{v:,.6f}".rstrip("0").rstrip(".")))
    # the decoder scores by seed mean, the tie rule
    da, db = a[decoder][scores], b[decoder][scores]
    n_seeds = da.index.get_level_values("seed").nunique()
    mean_a, mean_b = da.groupby(level="split").mean(), db.groupby(level="split").mean()
    sd_a = da.groupby(level="split").std(ddof=1).fillna(0.0)    # one seed = no spread
    sd_b = db.groupby(level="split").std(ddof=1).fillna(0.0)
    mean_gap = (mean_a - mean_b).abs().where(~(mean_a.isna() & mean_b.isna()), 0.0)
    mean_allowed = (2 * pd.concat([sd_a, sd_b]).groupby(level="split").max()).clip(lower=DECODER_MEAN_TOLERANCE)
    mean_off = mean_gap.isna() | mean_gap.gt(mean_allowed)
    by_mean = pd.concat({c: pd.DataFrame({"part two mean": mean_a[c], "part two sd": sd_a[c],
                                          "part four mean": mean_b[c], "part four sd": sd_b[c],
                                          "gap": mean_gap[c], "allowed": mean_allowed[c], "off": mean_off[c]})
                         for c in scores}, names=["column"])
    print(f"\ndecoder scores by seed mean, {n_seeds} seeds, {len(by_mean)} means, a gap passes when it is at "
          f"most {DECODER_MEAN_TOLERANCE:g} or at most twice the larger seed spread (the tie rule)")
    print(by_mean.to_string(float_format=lambda v: f"{v:,.6f}".rstrip("0").rstrip(".")))
    worst = {"score": float(gap.loc[~decoder, scores].max().max()) if scores and (~decoder).any() else 0.0,
             "decoder": float(mean_gap.max().max()) if scores and decoder.any() else 0.0,
             "count": float(gap[counts].max().max()) if counts else 0.0}
    n_off, n_mean_off = int(off.to_numpy().sum()), int(mean_off.to_numpy().sum())
else:
    compared, by_mean = None, None
    worst, n_off, n_mean_off = {"score": float("nan"), "decoder": float("nan"), "count": float("nan")}, -1, -1
fine = same_rows and n_off == 0 and n_mean_off == 0
print(f"{len(a)} saved rows, {len(b)} repeated rows, {len(scores)} score columns (deterministic rows, largest "
      f"gap {worst['score']:.2e}, allowed {SCORE_TOLERANCE:g}; decoder seed means, largest gap "
      f"{worst['decoder']:.2e}, allowed {DECODER_MEAN_TOLERANCE:g} or twice the seed spread), {len(counts)} "
      f"count columns (largest gap {worst['count']:.2e}, allowed {COUNT_TOLERANCE:g})")
if not fine:
    print("\n" + "!" * 78)
    print("WARNING  THE CONTROL RUN DOES NOT MATCH THE SAVED RESULT OF PART TWO")
    if not same_rows:
        print(f"the rows differ, {sorted(set(a.index) ^ set(b.index))[:10]}")
    else:
        if n_off:
            print(f"{n_off} numbers are off, the first ones")
            print(compared[compared["off"]].head(10).to_string())
        if n_mean_off:
            print(f"{n_mean_off} decoder seed means are off")
            print(by_mean[by_mean["off"]].to_string())
    print("The run step or its inputs changed. Nothing below would be comparable with part two.")
    print("!" * 78)
    raise RuntimeError("the control run differs from part two, see the warning above")
print(f"the control agrees with part two, {len(a)} rows, deterministic score gaps at most {worst['score']:.2e} "
      f"(allowed {SCORE_TOLERANCE:g}), decoder seed-mean gaps at most {worst['decoder']:.2e} (tie rule), "
      f"count gaps at most {worst['count']:.2e} (allowed {COUNT_TOLERANCE:g})")
print(took(start, "control"))

In [ ]:
# SAR runs. The three students in Sentinel-1 mode on the four SAR datasets, the validation splits only. One
# process per student on the same GPU (score.score_parallel), each loads its student once and runs part two's
# run step unchanged, so every number is what a run on its own gives. The runs of a dataset start as soon as it
# is on the runtime disk - every dataset there goes into one score_parallel call, the ones still arriving into
# the next, so the GPU waits only while none of the datasets left is there. A dataset on neither the Hub nor
# Drive is named and its runs are MISSING. How many run at once follows from the free GPU memory and RAM against
# the estimated need of one run, printed with the reason, and a dataset that allows fewer at once runs in a
# later wave. A run whose summary is on Drive is skipped before any process starts. Every finished run prints
# its minutes and its peak GPU memory, the last line sets the wall time against the runs added up and against
# part three's minutes one run at a time. Everything the run step prints goes to one log per student in
# RES / "logs".
start = time.perf_counter()
if "student weights" in prefetch:
    prefetch.wait("student weights", by="SAR runs")
left = [d for d in SAR_DATASETS if d in SAR_TODO]
for name in SAR_DATASETS:
    if name not in left:
        print(f"{name}  every student run has its summary on Drive, nothing to run")
part_rows, part_times = [], []
while left:
    here = prefetch.first_ready(left, by="SAR runs")      # waits only while none of them is on the runtime disk
    left = [d for d in left if d not in here]
    sar = [d for d in here if prefetch.record(d)["ok"]]
    for name in here:
        if name not in sar:
            print(f"{name}  NOT AVAILABLE, its runs are skipped")
    if sar:
        rows, times = score.score_parallel(PREP, list(SAR_ENCODERS), sar, RES, token=HF_TOKEN,
                                           checkpoints=CHECKPOINTS, device=DEVICE, splits=SPLITS,
                                           fast_decoder=FAST_DECODER, compile_cache=COMPILE_CACHE)
        part_rows.append(rows)
        part_times.append(times)
part_rows = [rows for rows in part_rows if len(rows)]
rows_sar = pd.concat(part_rows, ignore_index=True) if part_rows else pd.DataFrame()
times_sar = (pd.concat(part_times, ignore_index=True) if len(part_times) > 1 else part_times[0] if part_times
             else pd.DataFrame(columns=score.TIMES_COLUMNS))
print(score.gain_line(times_sar, time.perf_counter() - start))
runs = sorted(set(rows_sar["run"])) if len(rows_sar) else []
print(f"{len(rows_sar)} summary rows from {len(runs)} runs")
for run in runs:
    print(f"  {run}")
missing = missing_runs(SAR_RUNS)
print(f"{len(missing)} of {len(SAR_RUNS)} runs without a summary on Drive")
for run in missing:
    print(f"  MISSING {run}, see FAILED or NOT AVAILABLE above")
print(took(start, "sar_runs"))

In [ ]:
# Aerial runs. The three students in aerial mode on the FLAIR-HUB validation set, one process per student on
# the same GPU as in the SAR cell, as soon as the aerial sets are on the runtime disk. DINOv3 SAT reading the
# same aerial RGB is a saved row of part three. A run whose summary is on Drive is skipped before any process
# starts. Then every item of the fetcher, and the seconds the GPU work waited for data against the time the
# data takes one after the other, as the old data cells fetched it before the first run.
start = time.perf_counter()
if "student weights" in prefetch:
    prefetch.wait("student weights", by="aerial runs")
aerial_here = AERIAL_DATASET in prefetch and prefetch.wait(AERIAL_DATASET, by="aerial runs")["ok"]
if AERIAL_DATASET not in prefetch:
    print(f"{AERIAL_DATASET}  every student run has its summary on Drive, nothing to run")
elif not aerial_here:
    print(f"{AERIAL_DATASET}  NOT AVAILABLE, its runs are skipped")
if aerial_here:
    rows_aerial, times_aerial = score.score_parallel(PREP, list(AERIAL_ENCODERS), [AERIAL_DATASET], RES,
                                                     token=HF_TOKEN, checkpoints=CHECKPOINTS, device=DEVICE,
                                                     splits=SPLITS, fast_decoder=FAST_DECODER,
                                                     compile_cache=COMPILE_CACHE)
else:
    rows_aerial, times_aerial = pd.DataFrame(), pd.DataFrame(columns=score.TIMES_COLUMNS)
print(score.gain_line(times_aerial, time.perf_counter() - start))
runs = sorted(set(rows_aerial["run"])) if len(rows_aerial) else []
print(f"{len(rows_aerial)} summary rows from {len(runs)} runs")
for run in runs:
    print(f"  {run}")
missing = missing_runs(AERIAL_RUNS)
print(f"{len(missing)} of {len(AERIAL_RUNS)} runs without a summary on Drive")
for run in missing:
    print(f"  MISSING {run}, see FAILED or NOT AVAILABLE above")
prefetch.close()                                 # every item is through, the streams of the notebook are back
print("\nevery item of the fetcher (started_s and ready_s count from the start of the fetcher)")
show(prefetch.summary().round(1))
TIMES["waited_for_data"] = sum(prefetch.waited.values())
print(prefetch.waited_line())
print(took(start, "aerial_runs"))

In [ ]:
# Table. One table of every dataset, head and model on the validation split. The saved rows of part two
# (satellite DINOv3 at layer 21, C-RADIOv4-H at layer 32) and of part three (TerraFM-B reading Sentinel-1,
# students M and S, DINOv3 SAT reading the aerial RGB) are read from the summary file of each run, part
# three's rows are checked against part three's own table, then the rows of the three students. A row is
# read by the IoU of the positive class for lakes, alpine glaciers and snow and by mIoU for forest and the
# aerial set (column score, as part three's table). The tables go to Drive.
start = time.perf_counter()
pd.set_option("display.width", 220, "display.precision", 3)
parts, absent = {}, {}
for source, folder, runs in (("part two", REF, SAVED_RUNS["part two"]), ("part three", PART3, SAVED_RUNS["part three"]),
                             ("part four", RES, SAR_RUNS + AERIAL_RUNS)):
    rows, absent[source] = score.read_runs(folder, runs, splits=SPLITS)
    parts[source] = rows.assign(source=source)
    print(f"{source:<10} {len(rows)} rows from {len(runs) - len(absent[source])} of {len(runs)} runs in {folder}")
    for run in absent[source]:
        print(f"  MISSING {run}")
own = pd.read_csv(PART3 / "stage_a_table.csv")
own = own[own["source"] == "part three"] if "source" in own.columns else own
differ = score.check_against(parts["part three"], own[own["split"].isin(SPLITS)], tolerance=SAVED_TOLERANCE)
if len(differ):
    print(differ.to_string(index=False))
    raise RuntimeError("part three's summaries differ from part three's own table, see above")
print(f"part three's {len(parts['part three'])} rows agree with its own table, stage_a_table.csv")
ORDER = ["dinov3-l-sat", "cradio-v4-h", "terrafm-b-s1", "dinov3-aerial", "student-m", "student-s", *STUDENTS]
COLUMNS = ["source", "dataset", "encoder", "head", "split", "seeds", "metric", "score", "score_sd", "mIoU",
           "mIoU_sd", "IoU_pos", "IoU_pos_sd", "accuracy", "run"]
table = score.reading_score(pd.concat(parts.values(), ignore_index=True)).reindex(columns=COLUMNS)
table["model"] = pd.Categorical(table["encoder"], categories=ORDER, ordered=True)
table = table.sort_values(["dataset", "head", "model"]).drop(columns="model").reset_index(drop=True)
print(f"\nevery dataset, head and model on the {', '.join(SPLITS)} split, {len(table)} rows (score = "
      "IoU of the positive class or mIoU, decoder = mean and SD over the seeds)")
print(table.drop(columns="run").to_string(index=False))
val = table[table["head"] != "majority"]
present = [e for e in ORDER if e in set(val["encoder"])]
side = val.pivot_table(index=["dataset", "head"], columns="encoder", values="mIoU", aggfunc="first")[present]
pos = val.pivot_table(index=["dataset", "head"], columns="encoder", values="score", aggfunc="first")[present]
print("\nvalidation splits, mIoU, the models side by side (decoder = mean over the seeds)")
print(side.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\nvalidation splits, IoU of the positive class for {list(POSITIVE_CLASS_DATASETS)}, mIoU for the others, "
      "the score the verdicts read (decoder = mean over the seeds)")
print(pos.to_string(float_format=lambda v: f"{v:.3f}"))
stage0.save_report(RES, stage_a_table=table, stage_a_val_table=side.reset_index(),
                   stage_a_val_table_pos=pos.reset_index())
print(f"\nsaved {RES / 'stage_a_table.csv'}, {RES / 'stage_a_val_table.csv'} and {RES / 'stage_a_val_table_pos.csv'}")
print(took(start, "table"))

In [ ]:
# Verdicts. The rule per row and the words, written by the code (score.judge and
# score.stage_a_word), for the questions 1 to 5 (score.QUESTIONS). A decoder row is a
# win when its seed mean is higher by more than 0.01 AND by more than twice the larger seed SD, a probe row
# when the gap is more than 0.02, a loss the same way round, else a tie. The word counts the 4 SAR
# validation decoder rows, the probe rows are tallied next to it and the aerial row is reported, neither
# changes the word. Every compared row and verdicts.json go to Drive, then the compile cache as one archive.
start = time.perf_counter()
verdicts = score.stage_a_verdicts(table, split=SPLITS[0])
compared_rows = score.verdict_rows(verdicts)
print ("the rule, ")
for key, text in verdicts["rule"].items():
    print(f"  {key:<12} {text}")
print()
for line in score.verdict_lines(verdicts):
    print(line)
print("\nevery compared row (gap = model - reference, needed = the margin the gap must pass)")
show(compared_rows[["question", "model", "reference", "dataset", "head", "role", "metric", "model_score",
                    "model_sd", "reference_score", "reference_sd", "gap", "needed", "outcome"]].round(4))
stage0.save_report(RES, verdicts=verdicts, stage_a_rows=compared_rows, times=TIMES)
print(f"\nsaved {RES / 'verdicts.json'} and {RES / 'stage_a_rows.csv'}")
if FAST_DECODER:                                 # after the results, nothing compiles any more
    try:
        score.save_compile_cache(COMPILE_CACHE, COMPILE_ARCHIVE)
    except OSError as err:
        print(f"compile cache not saved ({type(err).__name__}, {err}), the next session compiles again")
print(took(start, "verdicts"))

In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
